# 01 — Explore Reference Samples

**Project:** Sentinel-1 SAR monitoring of artisanal/small-scale gold mining in Southern Ghana

This notebook presents the reference-sample exploration used for the monitoring workflow. Sentinel-2 information was used as reference information for labelling; it was **not** used as a predictor in the machine-learning models.

### Reference dataset
- 1,425 labelled samples
- 720 galamsey samples
- 705 non-galamsey samples
- Six reference plots
- SAR predictors: VV, VH, VV–VH difference, VV/VH ratio, and NPI

> **Portfolio note:** The repository does not include the full labelled CSV by default. The cells below are written so the notebook can be rerun when the CSV is supplied locally.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data/reference_samples.csv")

if DATA.exists():
    df = pd.read_csv(DATA)
    print(f"Loaded {len(df):,} samples from {DATA}")
else:
    df = None
    print("Reference CSV not found. This notebook is presentation-ready; place the labelled CSV at ../data/reference_samples.csv to run the exploratory cells.")

In [ ]:
# Expected columns from the modelling workflow:
# plot_id, Class, VV, VH
#
# The derived predictors are calculated below from VV and VH.

def add_sar_features(frame, eps=1e-12):
    out = frame.copy()
    out["VV_VH_diff"] = out["VV"] - out["VH"]
    vv_lin = 10 ** (out["VV"] / 10.0)
    vh_lin = 10 ** (out["VH"] / 10.0)
    out["VV_VH_ratio"] = vv_lin / (vh_lin + eps)
    out["NPI"] = (vv_lin - vh_lin) / (vv_lin + vh_lin + eps)
    return out

if df is not None:
    df = add_sar_features(df)
    display(df.head())

In [ ]:
if df is not None:
    print("Class counts")
    display(df["Class"].value_counts().rename(index={0:"Non-galamsey", 1:"Galamsey"}).to_frame("samples"))

    if "plot_id" in df.columns:
        print("Samples by reference plot")
        display(pd.crosstab(df["plot_id"], df["Class"]))

In [ ]:
if df is not None:
    feature_cols = ["VV", "VH", "VV_VH_diff", "VV_VH_ratio", "NPI"]
    available = [c for c in feature_cols if c in df.columns]

    fig, axes = plt.subplots(len(available), 1, figsize=(9, 3 * len(available)))
    if len(available) == 1:
        axes = [axes]

    for ax, col in zip(axes, available):
        for cls, label in [(0, "Non-galamsey"), (1, "Galamsey")]:
            subset = df.loc[df["Class"] == cls, col].dropna()
            ax.hist(subset, bins=30, alpha=0.6, label=label)
        ax.set_title(col)
        ax.legend()
        ax.set_xlabel(col)
        ax.set_ylabel("Count")

    plt.tight_layout()
    plt.show()

## Interpretation

The modelling workflow uses the five SAR-derived predictors above. VV and VH are the original Sentinel-1 backscatter variables; the remaining three are engineered from them.

The feature-importance results documented for the project indicate that **VH and VV were the dominant predictors across the tree-based and linear models**, while NPI, VV–VH difference, and VV/VH contributed less strongly.

This notebook is intentionally exploratory: it does not claim that a histogram or marginal distribution alone establishes separability between the classes.